# **Initialization of data**

In [32]:
import pandas as pd
import re

# Read the Excel file into a DataFrame
df = pd.read_excel("salesforce_report.xlsx")

# **Checking data**

In [33]:
# Display the first 5 rows
print(df.head(5))

            Start Date    End Date Campaign Record Type Campaign categories  \
0  2024-06-08 00:00:00  21/08/2024                Event   Tech; Fundraising   
1  2024-03-09 00:00:00  25/09/2024                Event                Tech   
2  2024-06-08 00:00:00  21/08/2024                Event   Tech; Fundraising   
3                  NaN  27/02/2024                Event                Tech   
4                  NaN  19/03/2024                Event                Tech   

                                       Campaign Name                Full Name  \
0  [Propel event] Aplica a grants con confianza (...  Valentina Medrano Coley   
1  [Workshop] Fortalece tu historia de impacto (2...  Valentina Medrano Coley   
2  [Propel event] Aplica a grants con confianza (...           Milagros Luque   
3  [Workshop] Eleva tu fundraising con ChatGPT I ...           Milagros Luque   
4       [Workshop] Visibiliza tu causa con IA (2024)           Milagros Luque   

  Primary Affiliation: Account Name   

In [34]:
# Display DataFrame info
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2002 entries, 0 to 2001
Data columns (total 19 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Start Date                         1741 non-null   object 
 1   End Date                           1975 non-null   object 
 2   Campaign Record Type               2002 non-null   object 
 3   Campaign categories                1631 non-null   object 
 4   Campaign Name                      2002 non-null   object 
 5   Full Name                          2002 non-null   object 
 6   Primary Affiliation: Account Name  1753 non-null   object 
 7   Email                              2002 non-null   object 
 8   Billing Country                    300 non-null    object 
 9   Country Presence                   60 non-null     object 
 10  Country                            18 non-null     object 
 11  Country.1                          182 non-null    objec

#**Step 1: Deleting duplicated rows**

In [35]:
# Store original shape
original_rows = df.shape[0]

# Drop duplicated rows
df= df.drop_duplicates()

# Count how many rows were removed
removed_rows = original_rows - df.shape[0]
print(f"Total duplicated rows removed: {removed_rows}")

Total duplicated rows removed: 5


#**Step 2: Column Removal**

In [36]:
# Value's percentage per column with missing data
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print(missing_pct)

#NAME?                               100.000000
Country                               99.098648
Country Presence                      96.995493
Social Cause                          96.344517
Country.1                             90.886329
Billing Country                       85.077616
Origin                                80.620931
Campaign categories                   18.577867
Start Date                            13.019529
Primary Affiliation: Account Name     12.368553
End Date                               1.352028
Campaign Record Type                   0.000000
Email                                  0.000000
Campaign Name                          0.000000
Full Name                              0.000000
Attended                               0.000000
Registered                             0.000000
Recibe newsletter                      0.000000
Campaign Subtype                       0.000000
dtype: float64


We noticed a lot of missing data in some columns, so I decided to remove any column that had more than 80% missing values.

In [37]:
# Define threshold
threshold = 0.8

# Calculate missing value ratios
missing_ratio = df.isna().mean()

# Identify columns to drop and to keep
dropped_cols = missing_ratio[missing_ratio >= threshold].index.tolist()
kept_cols = missing_ratio[missing_ratio < threshold].index.tolist()

print("Columns dropped (missing > 80%):")
for col in dropped_cols:
    print(f" - {col} ({missing_ratio[col]*100:.2f}% missing)")

print("\nColumns kept:")
for col in kept_cols:
    print(f" - {col} ({missing_ratio[col]*100:.2f}% missing)")

# Drop the columns
df = df.loc[:, df.isna().mean() < threshold]

Columns dropped (missing > 80%):
 - Billing Country (85.08% missing)
 - Country Presence (97.00% missing)
 - Country (99.10% missing)
 - Country.1 (90.89% missing)
 - Origin (80.62% missing)
 - Social Cause (96.34% missing)
 - #NAME? (100.00% missing)

Columns kept:
 - Start Date (13.02% missing)
 - End Date (1.35% missing)
 - Campaign Record Type (0.00% missing)
 - Campaign categories (18.58% missing)
 - Campaign Name (0.00% missing)
 - Full Name (0.00% missing)
 - Primary Affiliation: Account Name (12.37% missing)
 - Email (0.00% missing)
 - Registered (0.00% missing)
 - Attended (0.00% missing)
 - Recibe newsletter (0.00% missing)
 - Campaign Subtype (0.00% missing)


In [38]:
# Checking missing data percentages again
missing_pct = df.isna().mean().sort_values(ascending=False) * 100
print(missing_pct)

Campaign categories                  18.577867
Start Date                           13.019529
Primary Affiliation: Account Name    12.368553
End Date                              1.352028
Campaign Record Type                  0.000000
Campaign Name                         0.000000
Full Name                             0.000000
Email                                 0.000000
Registered                            0.000000
Attended                              0.000000
Recibe newsletter                     0.000000
Campaign Subtype                      0.000000
dtype: float64


The missing data issue is still present, but we can’t delete these columns because they contain a lot of relevant information.

#**Step 3: Standardizing the Format of Full Names**

In [39]:
# We apply title case to the "Full Name" column, skipping the header row
df.loc[1:, "Full Name"] = df.loc[1:, "Full Name"].astype(str).str.title()


#**Step 4: Formatting Campaign Name**

In [40]:
# Function to clean Campaign Name
def clean_campaign_name(name):
    if pd.isna(name):
        return name
    # Remove square brackets and content inside
    name = re.sub(r"\[.*?\]", "", name).strip()
    # Remove parentheses and content inside
    name = re.sub(r"\(.*?\)", "", name).strip()
    return name

# Apply cleaning
df["Campaign Name"] = df["Campaign Name"].apply(clean_campaign_name)


### Campaign Name Cleaning

There are two parts of the `Campaign Name` field that we removed:

1. **Square Brackets** `[]` – These contained repeating information already stored in another column (`Campaign Subtype`), so they were removed.  
2. **Year in Parentheses** `()` – While the year could be useful, in this dataset all campaigns occur between September and October 2024, so it was considered irrelevant and removed.

*Note: In future datasets with a wider range of years, extracting the year into a separate column (`Year`) could be relevant.*


#**Step 5: Formatting Placeholder or Missing Values**

In [41]:
# Visualizing there are values that are not properly tagged in "Primary Affiliation: Account Name" column
col = "Primary Affiliation: Account Name"

# Define the replacements
dictionary = {
    "-": "Unknown",
    "ninguno": "None",
    "ninguna": "None"
}

# Count how many rows match any of the keys before replacement
mask = df[col].isin(dictionary.keys())
rows_before = mask.sum()

# Apply the replacements
df[col] = df[col].replace(dictionary)

print(f"Rows affected by replacement: {rows_before}")

Rows affected by replacement: 2


In [42]:
# Convert date columns to datetime format
date_cols = ["Start Date", "End Date"]

#  Function to parse and format dates
def parse_and_format_date(series):

    # Try converting to datetime with dayfirst=True
    parsed = pd.to_datetime(series, dayfirst=True, errors="coerce")

    # Convert valid dates to text in dd/mm/yyyy format
    formatted = parsed.dt.strftime("%d/%m/%Y")

    # Replace NaN with empty string
    formatted = formatted.fillna("")

    return formatted


# Apply to each column
for col in date_cols:
    if col in df.columns:
        df[col] = parse_and_format_date(df[col])
    else:
        print(f"Column not found: {col}")

#**Optional Functions**

In [43]:
# Optional:
# This code blocks fills missing values with 'Unknown' for all columns facilitating readability.
def fill_missing_with_unknown(df):

    # Replace empty strings with NaN
    df = df.replace(r'^\s*$', pd.NA, regex=True)

    # Fill all missing or null values with 'Unknown'
    df_filled = df.fillna("Unknown")

    return df_filled

""""
In this case, we use the function to fill missing values for readibility purposes in the final dataset.
But if you prefer to keep missing values as NaN (usually better for analysis data),
you can kindly skip this step.
"""
# Executing the function
df =fill_missing_with_unknown(df)


We applied this cleaning function for better readability in the dashboard. If it will be used for IA/ML is a better option to keep it with NA Values (i.e., don't execute it).

In [44]:
""""
Optional:
As well, you can convert boolean columns from 1/0 to "True"/"False" facilitating readability
with this code block.
"""
""""

boolean_cols = ["Registered", "Attended", "Recibe newsletter"]
for col in boolean_cols:
    df[col] = df[col].map({1: "True", 0: "False"})

"""

'"\n\nboolean_cols = ["Registered", "Attended", "Recibe newsletter"]\nfor col in boolean_cols:\n    df[col] = df[col].map({1: "True", 0: "False"})\n    \n'

If a person who don't know what a boolean is, maybe it can result confusing interpreting 1 as True and 0 for False. However, we didn't execute this code block this time.

#**Final Result**

In [45]:
# Final cleaned DataFrame
cleaned_df =df.to_excel("cleaned_salesforce_report.xlsx", index=False)